In [ ]:
import heapq
from bisect import bisect_right
from collections import OrderedDict, defaultdict, deque

# Stack, Binary Search, Linked List

---
# 1) Introduction

Three structures whose questions reward knowing one template each by heart.

| Topic | Template to memorise | Typical cue |
|---|---|---|
| Stack | push/pop matching; **monotonic stack** for "next greater/smaller" | brackets, nested structure, "next warmer day", histogram |
| Binary search | `lo, hi` with a monotone predicate, **search on the answer** | sorted input, rotated array, "minimum capacity / speed so that …" |
| Linked list | dummy head, slow/fast pointers, in-place reversal | middle, cycle, k-th from end, reorder, merge |

**How to use each question block.** Read the *Problem* line, cover the rest, and try for five
minutes. If stuck, read only the *Signal*, try again, then the *Idea*. Run the cell; the
`assert` lines are LeetCode's examples plus edge cases. Tiers: ★★★ Blind 75 (do first),
★★ NeetCode 150 / Grind 169 / Top Interview 150, ★ other frequently asked.

---
# 2) Stack

A stack answers "what is the most recent unmatched thing?". Two families:

1. **Matching / evaluation** — brackets, expressions, path components, nested decoding.
2. **Monotonic stack** — keep indices whose values are increasing (or decreasing). When a
   new element breaks the order, every popped element has just found its *next greater
   (smaller) element*. Each index is pushed and popped once → $O(n)$.

```python
stack = []                         # indices, values decreasing
for i, x in enumerate(nums):
    while stack and nums[stack[-1]] < x:
        j = stack.pop()            # x is the next greater element of nums[j]
    stack.append(i)
```

## 2.1 Matching and evaluation

### 2.1.1 [20. Valid Parentheses](https://leetcode.com/problems/valid-parentheses/) — ★★★ · Easy

**Problem.** Is a string of `()[]{}` correctly matched and nested?  
**Signal.** Most recent open bracket must close first → stack.  
**Idea.** Push opening brackets. On a closing bracket the top must be its partner, else invalid. The stack must be empty at the end.  
**Cost.** O(n) time · O(n) space

In [ ]:
PAIRS = {")": "(", "]": "[", "}": "{"}


def is_valid(s: str) -> bool:
    """True if brackets are balanced and properly nested."""
    stack = []
    for ch in s:
        if ch in PAIRS:
            if not stack or stack.pop() != PAIRS[ch]:
                return False
        else:
            stack.append(ch)
    return not stack


assert is_valid("()[]{}")
assert not is_valid("(]")
assert is_valid("([])")
assert not is_valid("([)]")
assert not is_valid("(")
print(is_valid("{[()]}"))

### 2.1.2 [155. Min Stack](https://leetcode.com/problems/min-stack/) — ★★ · Medium

**Problem.** Stack with `push`, `pop`, `top` and `get_min`, all O(1).  
**Signal.** Minimum must survive pops → store the running minimum alongside each element.  
**Idea.** Each stack entry is `(value, min_so_far)`. The minimum below any entry never changes while that entry is on the stack, so popping restores the old minimum automatically.  
**Cost.** O(1) per operation · O(n) space

In [ ]:
class MinStack:
    """Stack that reports its minimum in O(1)."""

    def __init__(self) -> None:
        self.items: list[tuple[int, int]] = []

    def push(self, val: int) -> None:
        low = min(val, self.items[-1][1]) if self.items else val
        self.items.append((val, low))

    def pop(self) -> None:
        self.items.pop()

    def top(self) -> int:
        return self.items[-1][0]

    def get_min(self) -> int:
        return self.items[-1][1]


st = MinStack()
for v in [-2, 0, -3]:
    st.push(v)
assert st.get_min() == -3
st.pop()
assert st.top() == 0 and st.get_min() == -2
print(st.get_min())

### 2.1.3 [150. Evaluate Reverse Polish Notation](https://leetcode.com/problems/evaluate-reverse-polish-notation/) — ★★ · Medium

**Problem.** Evaluate a postfix expression of integers and `+ - * /` (division truncates toward 0).  
**Signal.** Postfix → operands on a stack, operator pops two.  
**Idea.** Push numbers. On an operator pop `b` then `a` (order matters for `-` and `/`) and push `a op b`. Use `int(a / b)` for truncation toward zero; `//` floors negatives.  
**Cost.** O(n) time · O(n) space

In [ ]:
def eval_rpn(tokens: list[str]) -> int:
    """Value of a postfix expression."""
    stack: list[int] = []
    for tok in tokens:
        if tok in "+-*/":
            b, a = stack.pop(), stack.pop()
            if tok == "+":
                stack.append(a + b)
            elif tok == "-":
                stack.append(a - b)
            elif tok == "*":
                stack.append(a * b)
            else:
                stack.append(int(a / b))
        else:
            stack.append(int(tok))
    return stack[0]


assert eval_rpn(["2", "1", "+", "3", "*"]) == 9
assert eval_rpn(["4", "13", "5", "/", "+"]) == 6
tokens = ["10", "6", "9", "3", "+", "-11", "*", "/", "*", "17", "+", "5", "+"]
assert eval_rpn(tokens) == 22
print(eval_rpn(tokens))

### 2.1.4 [232. Implement Queue using Stacks](https://leetcode.com/problems/implement-queue-using-stacks/) — ★★ · Easy

**Problem.** FIFO queue using only two stacks; amortised O(1) operations.  
**Signal.** Reversing a stack once gives queue order → inbox/outbox.  
**Idea.** Push onto `inbox`. To pop/peek, if `outbox` is empty, move everything from `inbox` to `outbox` (reversing it). Each element moves at most once → amortised O(1).  
**Cost.** amortised O(1) per op

In [ ]:
class MyQueue:
    """Queue built from two stacks."""

    def __init__(self) -> None:
        self.inbox: list[int] = []
        self.outbox: list[int] = []

    def push(self, x: int) -> None:
        self.inbox.append(x)

    def _shift(self) -> None:
        if not self.outbox:
            while self.inbox:
                self.outbox.append(self.inbox.pop())

    def pop(self) -> int:
        self._shift()
        return self.outbox.pop()

    def peek(self) -> int:
        self._shift()
        return self.outbox[-1]

    def empty(self) -> bool:
        return not self.inbox and not self.outbox


q = MyQueue()
q.push(1)
q.push(2)
assert q.peek() == 1 and q.pop() == 1 and not q.empty()
print(q.pop(), q.empty())

### 2.1.5 [225. Implement Stack using Queues](https://leetcode.com/problems/implement-stack-using-queues/) — ★ · Easy

**Problem.** LIFO stack using only queue operations.  
**Signal.** Rotate the queue after each push so the newest element is at the front.  
**Idea.** After appending `x`, move the `n − 1` older elements from the front to the back. Push is O(n), pop/top are O(1).  
**Cost.** O(n) push · O(1) pop

In [ ]:
class MyStack:
    """Stack built from one queue."""

    def __init__(self) -> None:
        self.q: deque[int] = deque()

    def push(self, x: int) -> None:
        self.q.append(x)
        for _ in range(len(self.q) - 1):
            self.q.append(self.q.popleft())

    def pop(self) -> int:
        return self.q.popleft()

    def top(self) -> int:
        return self.q[0]

    def empty(self) -> bool:
        return not self.q


s = MyStack()
s.push(1)
s.push(2)
assert s.top() == 2 and s.pop() == 2 and not s.empty()
print(s.pop(), s.empty())

### 2.1.6 [71. Simplify Path](https://leetcode.com/problems/simplify-path/) — ★★ · Medium

**Problem.** Canonicalise a Unix path (handle `.`, `..`, repeated slashes).  
**Signal.** Directory nesting → stack of path components.  
**Idea.** Split on `/`. Skip empty parts and `.`; `..` pops (if possible); anything else is pushed. Join with `/` and prefix `/`.  
**Cost.** O(n) time · O(n) space

In [ ]:
def simplify_path(path: str) -> str:
    """Canonical absolute path."""
    stack = []
    for part in path.split("/"):
        if part == "..":
            if stack:
                stack.pop()
        elif part and part != ".":
            stack.append(part)
    return "/" + "/".join(stack)


assert simplify_path("/home/") == "/home"
assert simplify_path("/home//foo/") == "/home/foo"
assert simplify_path("/home/user/Documents/../Pictures") == "/home/user/Pictures"
assert simplify_path("/../") == "/"
assert simplify_path("/.../a/../b/c/../d/./") == "/.../b/d"
print(simplify_path("/a/./b/../../c/"))

### 2.1.7 [227. Basic Calculator II](https://leetcode.com/problems/basic-calculator-ii/) — ★★ · Medium

**Problem.** Evaluate `+ - * /` with non-negative integers and spaces, no parentheses.  
**Signal.** Precedence without parentheses → apply `*`/`/` immediately, defer `+`/`−` on a stack.  
**Idea.** Read a number, then look at the operator *before* it (`op`). `+` pushes `num`, `−` pushes `−num`, `*`/`/` combine with the top of the stack. The answer is the stack sum.  
**Cost.** O(n) time · O(n) space (O(1) by keeping only the last term)

In [ ]:
def calculate_ii(s: str) -> int:
    """Evaluate an expression with + - * /."""
    stack: list[int] = []
    num, op = 0, "+"
    for i, ch in enumerate(s):
        if ch.isdigit():
            num = num * 10 + int(ch)
        if ch in "+-*/" or i == len(s) - 1:
            if op == "+":
                stack.append(num)
            elif op == "-":
                stack.append(-num)
            elif op == "*":
                stack.append(stack.pop() * num)
            else:
                stack.append(int(stack.pop() / num))
            num, op = 0, ch
    return sum(stack)


assert calculate_ii("3+2*2") == 7
assert calculate_ii(" 3/2 ") == 1
assert calculate_ii(" 3+5 / 2 ") == 5
assert calculate_ii("14-3/2") == 13
print(calculate_ii("3+2*2"))

### 2.1.8 [224. Basic Calculator](https://leetcode.com/problems/basic-calculator/) — ★★ · Hard

**Problem.** Evaluate `+`, `−`, parentheses and unary minus.  
**Signal.** Only `+`/`−` → running sum with a sign; parentheses → push context.  
**Idea.** Keep `result` and `sign`. On `(`, push the current `result` and `sign` and start fresh. On `)`, finish the inner result, multiply by the saved sign, add to the saved result.  
**Cost.** O(n) time · O(depth) space

In [ ]:
def calculate(s: str) -> int:
    """Evaluate an expression with + - and parentheses."""
    stack: list[int] = []
    result, num, sign = 0, 0, 1
    for ch in s:
        if ch.isdigit():
            num = num * 10 + int(ch)
        elif ch in "+-":
            result += sign * num
            num, sign = 0, 1 if ch == "+" else -1
        elif ch == "(":
            stack.extend([result, sign])
            result, sign = 0, 1
        elif ch == ")":
            result += sign * num
            num = 0
            saved_sign, saved_result = stack.pop(), stack.pop()
            result = saved_result + saved_sign * result
    return result + sign * num


assert calculate("1 + 1") == 2
assert calculate(" 2-1 + 2 ") == 3
assert calculate("(1+(4+5+2)-3)+(6+8)") == 23
assert calculate("-(2+3)") == -5
print(calculate("(1+(4+5+2)-3)+(6+8)"))

### 2.1.9 [394. Decode String](https://leetcode.com/problems/decode-string/) — ★★ · Medium

**Problem.** Decode `k[encoded]` (nested), e.g. `3[a2[c]]` → `accaccacc`.  
**Signal.** Nested brackets → stack of (previous string, repeat count).  
**Idea.** On `[` push the string built so far and the pending count, then reset. On `]` pop and set `current = prev + count * current`. Digits accumulate into `count` (multi-digit).  
**Cost.** O(output length) time

In [ ]:
def decode_string(s: str) -> str:
    """Expand k[...] encodings."""
    stack: list[tuple[str, int]] = []
    current, count = "", 0
    for ch in s:
        if ch.isdigit():
            count = count * 10 + int(ch)
        elif ch == "[":
            stack.append((current, count))
            current, count = "", 0
        elif ch == "]":
            prev, k = stack.pop()
            current = prev + current * k
        else:
            current += ch
    return current


assert decode_string("3[a]2[bc]") == "aaabcbc"
assert decode_string("3[a2[c]]") == "accaccacc"
assert decode_string("2[abc]3[cd]ef") == "abcabccdcdcdef"
assert decode_string("10[a]") == "a" * 10
print(decode_string("3[a2[c]]"))

### 2.1.10 [735. Asteroid Collision](https://leetcode.com/problems/asteroid-collision/) — ★★ · Medium

**Problem.** Asteroids move right (+) or left (−); colliding ones: smaller explodes, equal both explode. Final state?  
**Signal.** A left-mover only meets the most recent right-movers → stack.  
**Idea.** For each asteroid `a < 0`, while the top is a right-mover smaller than `|a|`, pop it. If the top equals `|a|`, pop and drop `a`. If the top is bigger, drop `a`. Otherwise push.  
**Cost.** O(n) time · O(n) space

In [ ]:
def asteroid_collision(asteroids: list[int]) -> list[int]:
    """Surviving asteroids after all collisions."""
    stack: list[int] = []
    for a in asteroids:
        alive = True
        while alive and a < 0 and stack and stack[-1] > 0:
            if stack[-1] < -a:
                stack.pop()
                continue
            if stack[-1] == -a:
                stack.pop()
            alive = False
        if alive:
            stack.append(a)
    return stack


assert asteroid_collision([5, 10, -5]) == [5, 10]
assert asteroid_collision([8, -8]) == []
assert asteroid_collision([10, 2, -5]) == [10]
assert asteroid_collision([-2, -1, 1, 2]) == [-2, -1, 1, 2]
print(asteroid_collision([5, 10, -5]))

### 2.1.11 [1249. Minimum Remove to Make Valid Parentheses](https://leetcode.com/problems/minimum-remove-to-make-valid-parentheses/) — ★★ · Medium

**Problem.** Remove the fewest parentheses so the string is valid; return any valid result.  
**Signal.** Unmatched `)` are found left-to-right; unmatched `(` are what remain on the stack.  
**Idea.** Scan with a stack of indices of `(`. A `)` with an empty stack is marked for removal. After the scan, every index still on the stack is an unmatched `(` and is removed too.  
**Cost.** O(n) time · O(n) space

In [ ]:
def min_remove_to_make_valid(s: str) -> str:
    """s with the fewest parentheses deleted to balance it."""
    remove, stack = set(), []
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        elif ch == ")":
            if stack:
                stack.pop()
            else:
                remove.add(i)
    remove.update(stack)
    return "".join(ch for i, ch in enumerate(s) if i not in remove)


assert min_remove_to_make_valid("lee(t(c)o)de)") == "lee(t(c)o)de"
assert min_remove_to_make_valid("a)b(c)d") == "ab(c)d"
assert min_remove_to_make_valid("))((") == ""
print(min_remove_to_make_valid("lee(t(c)o)de)"))

### 2.1.12 [921. Minimum Add to Make Parentheses Valid](https://leetcode.com/problems/minimum-add-to-make-parentheses-valid/) — ★ · Medium

**Problem.** Fewest insertions to balance a string of `(` and `)`.  
**Signal.** Only counts matter → a balance counter instead of a stack.  
**Idea.** Track `open` = unmatched `(`. A `)` with `open == 0` needs an inserted `(`. At the end, each unmatched `(` needs a `)`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def min_add_to_make_valid(s: str) -> int:
    """Number of insertions needed to balance s."""
    open_count = inserts = 0
    for ch in s:
        if ch == "(":
            open_count += 1
        elif open_count:
            open_count -= 1
        else:
            inserts += 1
    return inserts + open_count


assert min_add_to_make_valid("())") == 1
assert min_add_to_make_valid("(((") == 3
assert min_add_to_make_valid("()))((") == 4
print(min_add_to_make_valid("())"))

### 2.1.13 [32. Longest Valid Parentheses](https://leetcode.com/problems/longest-valid-parentheses/) — ★★ · Hard

**Problem.** Length of the longest well-formed parentheses substring.  
**Signal.** Stack of indices with a sentinel marking the last invalid position.  
**Idea.** Push `-1` as the base. `(` pushes its index. `)` pops; if the stack becomes empty, this `)` is unmatched and becomes the new base; otherwise the valid run ending here has length `i − stack[-1]`.  
**Cost.** O(n) time · O(n) space

In [ ]:
def longest_valid_parentheses(s: str) -> int:
    """Longest balanced parentheses substring."""
    stack, best = [-1], 0
    for i, ch in enumerate(s):
        if ch == "(":
            stack.append(i)
        else:
            stack.pop()
            if not stack:
                stack.append(i)
            else:
                best = max(best, i - stack[-1])
    return best


assert longest_valid_parentheses("(()") == 2
assert longest_valid_parentheses(")()())") == 4
assert longest_valid_parentheses("") == 0
assert longest_valid_parentheses("()(())") == 6
print(longest_valid_parentheses(")()())"))

### 2.1.14 [1047. Remove All Adjacent Duplicates In String](https://leetcode.com/problems/remove-all-adjacent-duplicates-in-string/) — ★ · Easy

**Problem.** Repeatedly delete adjacent equal pairs; return the final string.  
**Signal.** Removal can expose a new adjacent pair → stack of kept characters.  
**Idea.** If the incoming character equals the top, pop (the pair cancels); else push.  
**Cost.** O(n) time · O(n) space

In [ ]:
def remove_duplicates_str(s: str) -> str:
    """s after cancelling adjacent equal pairs."""
    stack: list[str] = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)


assert remove_duplicates_str("abbaca") == "ca"
assert remove_duplicates_str("azxxzy") == "ay"
print(remove_duplicates_str("abbaca"))

### 2.1.15 [2390. Removing Stars From a String](https://leetcode.com/problems/removing-stars-from-a-string/) — ★ · Medium

**Problem.** Each `*` deletes the closest non-star character to its left; return the result.  
**Signal.** "Closest to the left" = top of a stack.  
**Idea.** Push letters; a `*` pops.  
**Cost.** O(n) time · O(n) space

In [ ]:
def remove_stars(s: str) -> str:
    """s after applying star deletions."""
    stack: list[str] = []
    for ch in s:
        if ch == "*":
            stack.pop()
        else:
            stack.append(ch)
    return "".join(stack)


assert remove_stars("leet**cod*e") == "lecoe"
assert remove_stars("erase*****") == ""
print(remove_stars("leet**cod*e"))

### 2.1.16 [895. Maximum Frequency Stack](https://leetcode.com/problems/maximum-frequency-stack/) — ★★ · Hard

**Problem.** `push(x)`; `pop()` removes the most frequent element, ties broken by most recent.  
**Signal.** Group elements by frequency; each frequency level is its own stack.  
**Idea.** `freq[x]` counts occurrences; `group[f]` is a stack of elements that reached frequency `f`. Pushing `x` with new frequency `f` appends to `group[f]`. Pop from `group[max_freq]`; if it empties, `max_freq −= 1`. Copies at lower levels keep earlier frequencies intact.  
**Cost.** O(1) per op · O(n) space

In [ ]:
class FreqStack:
    """Stack popping the most frequent (then most recent) element."""

    def __init__(self) -> None:
        self.freq: dict[int, int] = defaultdict(int)
        self.group: dict[int, list[int]] = defaultdict(list)
        self.max_freq = 0

    def push(self, val: int) -> None:
        self.freq[val] += 1
        f = self.freq[val]
        self.max_freq = max(self.max_freq, f)
        self.group[f].append(val)

    def pop(self) -> int:
        val = self.group[self.max_freq].pop()
        self.freq[val] -= 1
        if not self.group[self.max_freq]:
            self.max_freq -= 1
        return val


fs = FreqStack()
for v in [5, 7, 5, 7, 4, 5]:
    fs.push(v)
assert [fs.pop() for _ in range(4)] == [5, 7, 5, 4]
print("ok")

## 2.2 Monotonic stack

### 2.2.1 [739. Daily Temperatures](https://leetcode.com/problems/daily-temperatures/) — ★★ · Medium

**Problem.** For each day, how many days until a warmer temperature (0 if never)?  
**Signal.** "Next greater element" → monotonic decreasing stack of indices.  
**Idea.** Keep indices of days still waiting for a warmer day; their temperatures decrease down the stack. A warmer day pops every cooler waiting day and writes the distance.  
**Cost.** O(n) time · O(n) space

In [ ]:
def daily_temperatures(temperatures: list[int]) -> list[int]:
    """Days to wait for a warmer temperature."""
    out = [0] * len(temperatures)
    stack: list[int] = []
    for i, t in enumerate(temperatures):
        while stack and temperatures[stack[-1]] < t:
            j = stack.pop()
            out[j] = i - j
        stack.append(i)
    return out


assert daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]) == [1, 1, 4, 2, 1, 1, 0, 0]
assert daily_temperatures([30, 60, 90]) == [1, 1, 0]
print(daily_temperatures([73, 74, 75, 71, 69, 72, 76, 73]))

### 2.2.2 [496. Next Greater Element I](https://leetcode.com/problems/next-greater-element-i/) — ★ · Easy

**Problem.** For each value in `nums1` (a subset of `nums2`), the next greater value to its right in `nums2`, or −1.  
**Signal.** Precompute next-greater for all of `nums2` with a monotonic stack, then look up.  
**Idea.** Run the Daily Temperatures loop on `nums2` storing `next_greater[value]`; answer each query from the map.  
**Cost.** O(m + n) time · O(n) space

In [ ]:
def next_greater_element(nums1: list[int], nums2: list[int]) -> list[int]:
    """Next greater value in nums2 for each query value."""
    nxt: dict[int, int] = {}
    stack: list[int] = []
    for x in nums2:
        while stack and stack[-1] < x:
            nxt[stack.pop()] = x
        stack.append(x)
    return [nxt.get(x, -1) for x in nums1]


assert next_greater_element([4, 1, 2], [1, 3, 4, 2]) == [-1, 3, -1]
assert next_greater_element([2, 4], [1, 2, 3, 4]) == [3, -1]
print(next_greater_element([4, 1, 2], [1, 3, 4, 2]))

### 2.2.3 [503. Next Greater Element II](https://leetcode.com/problems/next-greater-element-ii/) — ★ · Medium

**Problem.** Next greater element in a *circular* array.  
**Signal.** Circular → iterate over the array twice (indices `i % n`).  
**Idea.** Same monotonic stack, looping `2n` times; only push indices on the first pass so each index gets answered at most once.  
**Cost.** O(n) time · O(n) space

In [ ]:
def next_greater_elements(nums: list[int]) -> list[int]:
    """Circular next greater element."""
    n = len(nums)
    out = [-1] * n
    stack: list[int] = []
    for i in range(2 * n):
        x = nums[i % n]
        while stack and nums[stack[-1]] < x:
            out[stack.pop()] = x
        if i < n:
            stack.append(i)
    return out


assert next_greater_elements([1, 2, 1]) == [2, -1, 2]
assert next_greater_elements([1, 2, 3, 4, 3]) == [2, 3, 4, -1, 4]
print(next_greater_elements([1, 2, 1]))

### 2.2.4 [901. Online Stock Span](https://leetcode.com/problems/online-stock-span/) — ★ · Medium

**Problem.** For each new price, how many consecutive days (including today) had price ≤ today?  
**Signal.** Previous greater element, online → monotonic stack of (price, span).  
**Idea.** Pop every entry with price ≤ today and add its span to today's span — those days are covered by today forever. Push `(price, span)`.  
**Cost.** amortised O(1) per call

In [ ]:
class StockSpanner:
    """Online stock span."""

    def __init__(self) -> None:
        self.stack: list[tuple[int, int]] = []

    def next(self, price: int) -> int:
        span = 1
        while self.stack and self.stack[-1][0] <= price:
            span += self.stack.pop()[1]
        self.stack.append((price, span))
        return span


sp = StockSpanner()
out = [sp.next(p) for p in [100, 80, 60, 70, 60, 75, 85]]
assert out == [1, 1, 1, 2, 1, 4, 6]
print(out)

### 2.2.5 [853. Car Fleet](https://leetcode.com/problems/car-fleet/) — ★★ · Medium

**Problem.** Cars at `position[i]` with `speed[i]` drive to `target`; a faster car catching a slower one joins it. How many fleets arrive?  
**Signal.** Sort by position descending, compare arrival times → stack of fleet times.  
**Idea.** Process cars from closest to the target. Each car's arrival time is `(target − pos) / speed`. If it is ≤ the fleet ahead's time, it catches up and merges; otherwise it starts a new fleet.  
**Cost.** O(n log n) time · O(n) space

In [ ]:
def car_fleet(target: int, position: list[int], speed: list[int]) -> int:
    """Number of fleets reaching the target."""
    fleets: list[float] = []
    for pos, spd in sorted(zip(position, speed), reverse=True):
        time = (target - pos) / spd
        if not fleets or time > fleets[-1]:
            fleets.append(time)
    return len(fleets)


assert car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]) == 3
assert car_fleet(10, [3], [3]) == 1
assert car_fleet(100, [0, 2, 4], [4, 2, 1]) == 1
print(car_fleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))

### 2.2.6 [84. Largest Rectangle in Histogram](https://leetcode.com/problems/largest-rectangle-in-histogram/) — ★★ · Hard

**Problem.** Largest rectangle area in a histogram.  
**Signal.** Each bar's widest rectangle extends to the previous and next *smaller* bars → monotonic increasing stack.  
**Idea.** Keep indices with increasing heights. When bar `i` is lower than the top, the top's rectangle ends at `i`; its left boundary is the new top of the stack after popping. Append a 0-height sentinel to flush the stack at the end.  
**Cost.** O(n) time · O(n) space

In [ ]:
def largest_rectangle_area(heights: list[int]) -> int:
    """Largest rectangle in a histogram."""
    stack: list[int] = []
    best = 0
    for i, h in enumerate(heights + [0]):
        while stack and heights[stack[-1]] >= h:
            height = heights[stack.pop()]
            left = stack[-1] if stack else -1
            best = max(best, height * (i - left - 1))
        stack.append(i)
    return best


assert largest_rectangle_area([2, 1, 5, 6, 2, 3]) == 10
assert largest_rectangle_area([2, 4]) == 4
assert largest_rectangle_area([1, 1, 1, 1]) == 4
print(largest_rectangle_area([2, 1, 5, 6, 2, 3]))

### 2.2.7 [85. Maximal Rectangle](https://leetcode.com/problems/maximal-rectangle/) — ★ · Hard

**Problem.** Largest rectangle of 1s in a binary matrix.  
**Signal.** Each row turns into a histogram of consecutive 1s above → run Largest Rectangle per row.  
**Idea.** Maintain `heights[c]` = number of consecutive `"1"` ending at the current row in column `c` (reset to 0 on `"0"`). Apply the histogram algorithm to every row.  
**Cost.** O(rows · cols) time · O(cols) space

In [ ]:
def maximal_rectangle(matrix: list[list[str]]) -> int:
    """Largest all-ones rectangle area."""
    heights = [0] * len(matrix[0])
    best = 0
    for row in matrix:
        for c, cell in enumerate(row):
            heights[c] = heights[c] + 1 if cell == "1" else 0
        best = max(best, largest_rectangle_area(heights))
    return best


grid = [list("10100"), list("10111"), list("11111"), list("10010")]
assert maximal_rectangle(grid) == 6
assert maximal_rectangle([["0"]]) == 0
print(maximal_rectangle(grid))

### 2.2.8 [402. Remove K Digits](https://leetcode.com/problems/remove-k-digits/) — ★ · Medium

**Problem.** Remove `k` digits from a number string to make it as small as possible.  
**Signal.** Lexicographically smallest → greedy monotonic increasing stack.  
**Idea.** A digit followed by a smaller one should be removed (the earlier position matters more). While `k > 0` and the top is greater than the incoming digit, pop. Leftover `k` removes from the end. Strip leading zeros.  
**Cost.** O(n) time · O(n) space

In [ ]:
def remove_kdigits(num: str, k: int) -> str:
    """Smallest number after deleting k digits."""
    stack: list[str] = []
    for d in num:
        while k and stack and stack[-1] > d:
            stack.pop()
            k -= 1
        stack.append(d)
    if k:
        stack = stack[:-k]
    return "".join(stack).lstrip("0") or "0"


assert remove_kdigits("1432219", 3) == "1219"
assert remove_kdigits("10200", 1) == "200"
assert remove_kdigits("10", 2) == "0"
print(remove_kdigits("1432219", 3))

### 2.2.9 [316. Remove Duplicate Letters](https://leetcode.com/problems/remove-duplicate-letters/) — ★ · Medium

**Problem.** Remove duplicate letters so each appears once, giving the lexicographically smallest result.  
**Signal.** Remove K Digits plus "only pop a letter if it appears again later".  
**Idea.** Track the last index of each letter and which letters are in the stack. For each new letter not already used, pop larger letters from the top as long as they occur later.  
**Cost.** O(n) time · O(26) space

In [ ]:
def remove_duplicate_letters(s: str) -> str:
    """Smallest subsequence containing each letter once."""
    last = {ch: i for i, ch in enumerate(s)}
    stack: list[str] = []
    used: set[str] = set()
    for i, ch in enumerate(s):
        if ch in used:
            continue
        while stack and stack[-1] > ch and last[stack[-1]] > i:
            used.remove(stack.pop())
        stack.append(ch)
        used.add(ch)
    return "".join(stack)


assert remove_duplicate_letters("bcabc") == "abc"
assert remove_duplicate_letters("cbacdcbc") == "acdb"
print(remove_duplicate_letters("cbacdcbc"))

### 2.2.10 [907. Sum of Subarray Minimums](https://leetcode.com/problems/sum-of-subarray-minimums/) — ★ · Medium

**Problem.** Sum of `min(subarray)` over all subarrays, mod 1e9+7.  
**Signal.** Contribution technique: count subarrays where each element is the minimum.  
**Idea.** For `arr[i]`, let `left` = distance to the previous smaller element and `right` = distance to the next smaller-or-equal element (strict on one side avoids double counting ties). It is the minimum of `left × right` subarrays. One monotonic stack pass gives both distances.  
**Cost.** O(n) time · O(n) space

In [ ]:
def sum_subarray_mins(arr: list[int]) -> int:
    """Sum of minimums of all subarrays mod 1e9+7."""
    total = 0
    stack: list[int] = []
    extended = arr + [0]
    for i, x in enumerate(extended):
        while stack and extended[stack[-1]] >= x:
            j = stack.pop()
            left = j - (stack[-1] if stack else -1)
            right = i - j
            total += arr[j] * left * right
        stack.append(i)
    return total % (10**9 + 7)


assert sum_subarray_mins([3, 1, 2, 4]) == 17
assert sum_subarray_mins([11, 81, 94, 43, 3]) == 444
print(sum_subarray_mins([3, 1, 2, 4]))

### 2.2.11 [1762. Buildings With an Ocean View](https://leetcode.com/problems/buildings-with-an-ocean-view/) — ★ · Medium · Premium

**Problem.** Ocean on the right; return indices of buildings strictly taller than everything to their right.  
**Signal.** Scan from the right with a running max (or a decreasing stack from the left).  
**Idea.** Walk right to left, keeping the tallest height seen; a building taller than that has a view. Reverse the collected indices.  
**Cost.** O(n) time · O(1) extra space

In [ ]:
def find_buildings(heights: list[int]) -> list[int]:
    """Indices of buildings that see the ocean."""
    out, tallest = [], 0
    for i in range(len(heights) - 1, -1, -1):
        if heights[i] > tallest:
            out.append(i)
            tallest = heights[i]
    return out[::-1]


assert find_buildings([4, 2, 3, 1]) == [0, 2, 3]
assert find_buildings([4, 3, 2, 1]) == [0, 1, 2, 3]
assert find_buildings([1, 3, 2, 4]) == [3]
print(find_buildings([4, 2, 3, 1]))

---
# 3) Binary Search

One template avoids every off-by-one: search for the **first index where a monotone
predicate becomes true**.

```python
lo, hi = 0, n            # answer lies in [lo, hi]; hi = n means "never true"
while lo < hi:
    mid = (lo + hi) // 2
    if predicate(mid):
        hi = mid         # mid may be the answer
    else:
        lo = mid + 1     # mid is definitely not the answer
return lo
```

**Search on the answer.** When asked for "the minimum speed / capacity / days such that
something is feasible", the answer space is a range of integers and feasibility is
monotone (if speed `v` works, any faster speed works). Binary-search that range and test
feasibility with a greedy $O(n)$ check.

## 3.1 Classic binary search

### 3.1.1 [704. Binary Search](https://leetcode.com/problems/binary-search/) — ★★ · Easy

**Problem.** Index of `target` in a sorted array, or −1.  
**Signal.** Sorted + lookup → binary search.  
**Idea.** Find the first index with `nums[i] >= target`, then check equality.  
**Cost.** O(log n) time · O(1) space

In [ ]:
def search(nums: list[int], target: int) -> int:
    """Index of target in sorted nums, or -1."""
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo if lo < len(nums) and nums[lo] == target else -1


assert search([-1, 0, 3, 5, 9, 12], 9) == 4
assert search([-1, 0, 3, 5, 9, 12], 2) == -1
print(search([-1, 0, 3, 5, 9, 12], 9))

### 3.1.2 [35. Search Insert Position](https://leetcode.com/problems/search-insert-position/) — ★★ · Easy

**Problem.** Index of `target`, or where it would be inserted to keep order.  
**Signal.** The template's answer *is* the insertion point (lower bound).  
**Idea.** First index with `nums[i] >= target`. Equivalent to `bisect_left`.  
**Cost.** O(log n)

In [ ]:
def search_insert(nums: list[int], target: int) -> int:
    """Lower bound of target in nums."""
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] >= target:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert search_insert([1, 3, 5, 6], 5) == 2
assert search_insert([1, 3, 5, 6], 2) == 1
assert search_insert([1, 3, 5, 6], 7) == 4
print(search_insert([1, 3, 5, 6], 2))

### 3.1.3 [278. First Bad Version](https://leetcode.com/problems/first-bad-version/) — ★★ · Easy

**Problem.** Versions `1..n`; after the first bad one, all are bad. Find it with few API calls.  
**Signal.** Monotone predicate `is_bad(v)` → first-true binary search.  
**Idea.** The template with `predicate = is_bad`, range `[1, n]`.  
**Cost.** O(log n) calls

In [ ]:
def first_bad_version(n: int, is_bad) -> int:
    """First version for which is_bad is True."""
    lo, hi = 1, n
    while lo < hi:
        mid = (lo + hi) // 2
        if is_bad(mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


assert first_bad_version(5, lambda v: v >= 4) == 4
assert first_bad_version(1, lambda v: v >= 1) == 1
print(first_bad_version(2**31 - 1, lambda v: v >= 1702766719))

### 3.1.4 [34. Find First and Last Position of Element in Sorted Array](https://leetcode.com/problems/find-first-and-last-position-of-element-in-sorted-array/) — ★★ · Medium

**Problem.** `[first, last]` index of `target` in a sorted array, or `[-1, -1]`.  
**Signal.** Two binary searches: lower bound of `target` and lower bound of `target + 1`.  
**Idea.** First = first index with `nums[i] >= target`. Last = (first index with `nums[i] > target`) − 1. If first is out of range or not equal to `target`, it is absent.  
**Cost.** O(log n)

In [ ]:
def search_range(nums: list[int], target: int) -> list[int]:
    """First and last index of target."""

    def lower_bound(x: int) -> int:
        lo, hi = 0, len(nums)
        while lo < hi:
            mid = (lo + hi) // 2
            if nums[mid] >= x:
                hi = mid
            else:
                lo = mid + 1
        return lo

    first = lower_bound(target)
    if first == len(nums) or nums[first] != target:
        return [-1, -1]
    return [first, lower_bound(target + 1) - 1]


assert search_range([5, 7, 7, 8, 8, 10], 8) == [3, 4]
assert search_range([5, 7, 7, 8, 8, 10], 6) == [-1, -1]
assert search_range([], 0) == [-1, -1]
print(search_range([5, 7, 7, 8, 8, 10], 8))

### 3.1.5 [74. Search a 2D Matrix](https://leetcode.com/problems/search-a-2d-matrix/) — ★★ · Medium

**Problem.** Rows sorted, and each row starts after the previous row ends. Is `target` present?  
**Signal.** The matrix is one sorted array of length `m·n` in disguise.  
**Idea.** Binary search index `k` in `[0, m·n)`; the element is `matrix[k // n][k % n]`.  
**Cost.** O(log(mn))

In [ ]:
def search_matrix(matrix: list[list[int]], target: int) -> bool:
    """True if target is in the row-major sorted matrix."""
    rows, cols = len(matrix), len(matrix[0])
    lo, hi = 0, rows * cols - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        value = matrix[mid // cols][mid % cols]
        if value == target:
            return True
        if value < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return False


m = [[1, 3, 5, 7], [10, 11, 16, 20], [23, 30, 34, 60]]
assert search_matrix(m, 3)
assert not search_matrix(m, 13)
print(search_matrix(m, 34))

### 3.1.6 [240. Search a 2D Matrix II](https://leetcode.com/problems/search-a-2d-matrix-ii/) — ★ · Medium

**Problem.** Rows and columns sorted independently. Is `target` present?  
**Signal.** Start at the top-right corner: it splits the remaining matrix like a BST root.  
**Idea.** From top-right, if the value is too big, the whole column below is too big → move left; if too small, the whole row to the left is too small → move down.  
**Cost.** O(m + n) time · O(1) space

In [ ]:
def search_matrix_ii(matrix: list[list[int]], target: int) -> bool:
    """True if target is in a row- and column-sorted matrix."""
    r, c = 0, len(matrix[0]) - 1
    while r < len(matrix) and c >= 0:
        value = matrix[r][c]
        if value == target:
            return True
        if value > target:
            c -= 1
        else:
            r += 1
    return False


m = [
    [1, 4, 7, 11, 15],
    [2, 5, 8, 12, 19],
    [3, 6, 9, 16, 22],
    [10, 13, 14, 17, 24],
    [18, 21, 23, 26, 30],
]
assert search_matrix_ii(m, 5)
assert not search_matrix_ii(m, 20)
print(search_matrix_ii(m, 5))

### 3.1.7 [69. Sqrt(x)](https://leetcode.com/problems/sqrtx/) — ★★ · Easy

**Problem.** Integer square root (floor), no built-in power functions.  
**Signal.** Largest `r` with `r² ≤ x` → binary search on the answer.  
**Idea.** Find the first `r` with `r² > x` in `[0, x + 1]`, then subtract 1.  
**Cost.** O(log x)

In [ ]:
def my_sqrt(x: int) -> int:
    """floor(sqrt(x))."""
    lo, hi = 0, x + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if mid * mid > x:
            hi = mid
        else:
            lo = mid + 1
    return lo - 1


assert my_sqrt(4) == 2
assert my_sqrt(8) == 2
assert my_sqrt(0) == 0
assert my_sqrt(2147395599) == 46339
print(my_sqrt(8))

### 3.1.8 [367. Valid Perfect Square](https://leetcode.com/problems/valid-perfect-square/) — ★ · Easy

**Problem.** Is `num` a perfect square (no `sqrt`)?  
**Signal.** Same search as Sqrt(x), then check equality.  
**Idea.** Compute the integer square root by binary search and test `r * r == num`.  
**Cost.** O(log n)

In [ ]:
def is_perfect_square(num: int) -> bool:
    """True if num is a perfect square."""
    r = my_sqrt(num)
    return r * r == num


assert is_perfect_square(16)
assert not is_perfect_square(14)
print(is_perfect_square(808201))

### 3.1.9 [162. Find Peak Element](https://leetcode.com/problems/find-peak-element/) — ★★ · Medium

**Problem.** Return any index whose value is greater than its neighbours (`nums[-1] = nums[n] = −∞`).  
**Signal.** Binary search on the slope: go uphill.  
**Idea.** If `nums[mid] < nums[mid + 1]`, the slope rises to the right, and a peak must exist there (it either keeps rising to the edge, or turns down). Otherwise a peak is at `mid` or to its left.  
**Cost.** O(log n)

In [ ]:
def find_peak_element(nums: list[int]) -> int:
    """Index of some local maximum."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo


assert find_peak_element([1, 2, 3, 1]) == 2
assert find_peak_element([1, 2, 1, 3, 5, 6, 4]) in (1, 5)
print(find_peak_element([1, 2, 1, 3, 5, 6, 4]))

### 3.1.10 [540. Single Element in a Sorted Array](https://leetcode.com/problems/single-element-in-a-sorted-array/) — ★ · Medium

**Problem.** Sorted array where every element appears twice except one; find it in O(log n).  
**Signal.** Before the single element, pairs start at even indices; after it, at odd indices.  
**Idea.** Look at even `mid` (shift by one if odd). If `nums[mid] == nums[mid + 1]` the pairing is intact to the left, so the single element is to the right; else it is at `mid` or left.  
**Cost.** O(log n)

In [ ]:
def single_non_duplicate(nums: list[int]) -> int:
    """The element appearing once."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if mid % 2:
            mid -= 1
        if nums[mid] == nums[mid + 1]:
            lo = mid + 2
        else:
            hi = mid
    return nums[lo]


assert single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]) == 2
assert single_non_duplicate([3, 3, 7, 7, 10, 11, 11]) == 10
print(single_non_duplicate([1, 1, 2, 3, 3, 4, 4, 8, 8]))

### 3.1.11 [1539. Kth Missing Positive Number](https://leetcode.com/problems/kth-missing-positive-number/) — ★ · Easy

**Problem.** Sorted positive integers; return the `k`-th positive integer missing from the array.  
**Signal.** Missing count before index `i` is `arr[i] − (i + 1)`, which is monotone.  
**Idea.** Find the first index `lo` where missing count ≥ `k`. The answer is `lo + k` (the `lo` present numbers before it push the `k`-th missing number up by `lo`).  
**Cost.** O(log n)

In [ ]:
def find_kth_positive(arr: list[int], k: int) -> int:
    """k-th missing positive integer."""
    lo, hi = 0, len(arr)
    while lo < hi:
        mid = (lo + hi) // 2
        if arr[mid] - (mid + 1) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo + k


assert find_kth_positive([2, 3, 4, 7, 11], 5) == 9
assert find_kth_positive([1, 2, 3, 4], 2) == 6
print(find_kth_positive([2, 3, 4, 7, 11], 5))

### 3.1.12 [658. Find K Closest Elements](https://leetcode.com/problems/find-k-closest-elements/) — ★★ · Medium

**Problem.** The `k` values closest to `x` in a sorted array, returned sorted (ties → smaller).  
**Signal.** The answer is a window of length `k` → binary search the window's left edge.  
**Idea.** Search `left` in `[0, n − k]`. Compare `x − arr[mid]` with `arr[mid + k] − x`: if the left end is farther from `x` than the element just past the right end, shift the window right.  
**Cost.** O(log(n − k) + k)

In [ ]:
def find_closest_elements(arr: list[int], k: int, x: int) -> list[int]:
    """k closest values to x, sorted."""
    lo, hi = 0, len(arr) - k
    while lo < hi:
        mid = (lo + hi) // 2
        if x - arr[mid] > arr[mid + k] - x:
            lo = mid + 1
        else:
            hi = mid
    return arr[lo : lo + k]


assert find_closest_elements([1, 2, 3, 4, 5], 4, 3) == [1, 2, 3, 4]
assert find_closest_elements([1, 1, 2, 3, 4, 5], 4, -1) == [1, 1, 2, 3]
print(find_closest_elements([1, 2, 3, 4, 5], 4, 3))

### 3.1.13 [2300. Successful Pairs of Spells and Potions](https://leetcode.com/problems/successful-pairs-of-spells-and-potions/) — ★ · Medium

**Problem.** For each spell, count potions with `spell × potion ≥ success`.  
**Signal.** Sort potions once; each spell needs a threshold lookup.  
**Idea.** A spell `s` needs `potion ≥ ceil(success / s)`. With potions sorted, binary search the first such potion; the count is everything from there to the end.  
**Cost.** O((m + n) log m)

In [ ]:
def successful_pairs(spells: list[int], potions: list[int], success: int) -> list[int]:
    """Number of successful potions per spell."""
    potions = sorted(potions)
    m = len(potions)
    out = []
    for s in spells:
        need = (success + s - 1) // s
        lo, hi = 0, m
        while lo < hi:
            mid = (lo + hi) // 2
            if potions[mid] >= need:
                hi = mid
            else:
                lo = mid + 1
        out.append(m - lo)
    return out


assert successful_pairs([5, 1, 3], [1, 2, 3, 4, 5], 7) == [4, 0, 3]
assert successful_pairs([3, 1, 2], [8, 5, 8], 16) == [2, 0, 2]
print(successful_pairs([5, 1, 3], [1, 2, 3, 4, 5], 7))

## 3.2 Rotated arrays

### 3.2.1 [153. Find Minimum in Rotated Sorted Array](https://leetcode.com/problems/find-minimum-in-rotated-sorted-array/) — ★★★ · Medium

**Problem.** Minimum of a rotated sorted array of distinct values, O(log n).  
**Signal.** Compare `mid` with the right end to know which half is sorted.  
**Idea.** If `nums[mid] > nums[hi]`, the rotation point (the minimum) is to the right of `mid`. Otherwise `mid..hi` is sorted, so the minimum is at `mid` or to its left.  
**Cost.** O(log n)

In [ ]:
def find_min(nums: list[int]) -> int:
    """Minimum of a rotated sorted array."""
    lo, hi = 0, len(nums) - 1
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] > nums[hi]:
            lo = mid + 1
        else:
            hi = mid
    return nums[lo]


assert find_min([3, 4, 5, 1, 2]) == 1
assert find_min([4, 5, 6, 7, 0, 1, 2]) == 0
assert find_min([11, 13, 15, 17]) == 11
print(find_min([4, 5, 6, 7, 0, 1, 2]))

### 3.2.2 [33. Search in Rotated Sorted Array](https://leetcode.com/problems/search-in-rotated-sorted-array/) — ★★★ · Medium

**Problem.** Index of `target` in a rotated sorted array of distinct values, O(log n).  
**Signal.** One half around `mid` is always sorted → check whether `target` lies in that half.  
**Idea.** If `nums[lo] <= nums[mid]`, the left half is sorted: go left iff `nums[lo] <= target < nums[mid]`. Otherwise the right half is sorted: go right iff `nums[mid] < target <= nums[hi]`.  
**Cost.** O(log n)

In [ ]:
def search_rotated(nums: list[int], target: int) -> int:
    """Index of target in a rotated sorted array, or -1."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return -1


assert search_rotated([4, 5, 6, 7, 0, 1, 2], 0) == 4
assert search_rotated([4, 5, 6, 7, 0, 1, 2], 3) == -1
assert search_rotated([1], 0) == -1
assert search_rotated([3, 1], 1) == 1
print(search_rotated([4, 5, 6, 7, 0, 1, 2], 0))

### 3.2.3 [81. Search in Rotated Sorted Array II](https://leetcode.com/problems/search-in-rotated-sorted-array-ii/) — ★ · Medium

**Problem.** Same as above, but duplicates are allowed; return whether `target` exists.  
**Signal.** Duplicates can hide which half is sorted → when `nums[lo] == nums[mid] == nums[hi]`, shrink both ends.  
**Idea.** Same logic as the distinct version; the ambiguous case shrinks `lo` and `hi` by one, which makes the worst case O(n) (e.g. all equal values).  
**Cost.** O(log n) average, O(n) worst

In [ ]:
def search_rotated_ii(nums: list[int], target: int) -> bool:
    """True if target is in a rotated sorted array with duplicates."""
    lo, hi = 0, len(nums) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo, hi = lo + 1, hi - 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1
            else:
                hi = mid - 1
    return False


assert search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 0)
assert not search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 3)
assert search_rotated_ii([1, 0, 1, 1, 1], 0)
print(search_rotated_ii([2, 5, 6, 0, 0, 1, 2], 0))

## 3.3 Search on the answer

### 3.3.1 [875. Koko Eating Bananas](https://leetcode.com/problems/koko-eating-bananas/) — ★★ · Medium

**Problem.** Minimum eating speed `k` to finish all piles within `h` hours (one pile per hour max).  
**Signal.** "Minimum speed such that feasible" + monotone feasibility → binary search on `k`.  
**Idea.** Hours needed at speed `k` is `sum(ceil(p / k))`. Search `k` in `[1, max(piles)]` for the first speed with hours ≤ `h`.  
**Cost.** O(n log max) time

In [ ]:
def min_eating_speed(piles: list[int], h: int) -> int:
    """Slowest speed that finishes within h hours."""
    lo, hi = 1, max(piles)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((p + mid - 1) // mid for p in piles) <= h:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert min_eating_speed([3, 6, 7, 11], 8) == 4
assert min_eating_speed([30, 11, 23, 4, 20], 5) == 30
assert min_eating_speed([30, 11, 23, 4, 20], 6) == 23
print(min_eating_speed([3, 6, 7, 11], 8))

### 3.3.2 [1011. Capacity To Ship Packages Within D Days](https://leetcode.com/problems/capacity-to-ship-packages-within-d-days/) — ★★ · Medium

**Problem.** Minimum ship capacity to ship packages in order within `days` days.  
**Signal.** Minimum capacity such that feasible → binary search on capacity, greedy check.  
**Idea.** Capacity lies in `[max(weights), sum(weights)]`. Feasibility: load greedily, start a new day when the next package would overflow; count days.  
**Cost.** O(n log sum) time

In [ ]:
def ship_within_days(weights: list[int], days: int) -> int:
    """Least capacity shipping everything within days."""

    def days_needed(cap: int) -> int:
        count, load = 1, 0
        for w in weights:
            if load + w > cap:
                count += 1
                load = 0
            load += w
        return count

    lo, hi = max(weights), sum(weights)
    while lo < hi:
        mid = (lo + hi) // 2
        if days_needed(mid) <= days:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5) == 15
assert ship_within_days([3, 2, 2, 4, 1, 4], 3) == 6
assert ship_within_days([1, 2, 3, 1, 1], 4) == 3
print(ship_within_days([1, 2, 3, 4, 5, 6, 7, 8, 9, 10], 5))

### 3.3.3 [410. Split Array Largest Sum](https://leetcode.com/problems/split-array-largest-sum/) — ★ · Hard

**Problem.** Split into `k` non-empty contiguous parts minimising the largest part sum.  
**Signal.** Identical to Ship Within Days: "largest part ≤ cap" is feasible with ≤ k parts.  
**Idea.** Binary search `cap` in `[max, sum]`; greedy count of parts needed for `cap`.  
**Cost.** O(n log sum)

In [ ]:
def split_array(nums: list[int], k: int) -> int:
    """Minimised largest subarray sum over k parts."""
    return ship_within_days(nums, k)


assert split_array([7, 2, 5, 10, 8], 2) == 18
assert split_array([1, 2, 3, 4, 5], 2) == 9
print(split_array([7, 2, 5, 10, 8], 2))

### 3.3.4 [1482. Minimum Number of Days to Make m Bouquets](https://leetcode.com/problems/minimum-number-of-days-to-make-m-bouquets/) — ★ · Medium

**Problem.** Flower `i` blooms on day `bloom[i]`; a bouquet needs `k` *adjacent* bloomed flowers. Earliest day to make `m` bouquets?  
**Signal.** More days → more bloomed flowers → monotone → binary search on the day.  
**Idea.** For a day `d`, scan left to right counting consecutive bloomed flowers; every `k` in a row is a bouquet. Search the first `d` giving ≥ `m` bouquets. Impossible if `m·k > n`.  
**Cost.** O(n log max)

In [ ]:
def min_days(bloom_day: list[int], m: int, k: int) -> int:
    """Earliest day to form m bouquets of k adjacent flowers."""
    if m * k > len(bloom_day):
        return -1

    def bouquets(day: int) -> int:
        count = run = 0
        for b in bloom_day:
            run = run + 1 if b <= day else 0
            if run == k:
                count += 1
                run = 0
        return count

    lo, hi = min(bloom_day), max(bloom_day)
    while lo < hi:
        mid = (lo + hi) // 2
        if bouquets(mid) >= m:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert min_days([1, 10, 3, 10, 2], 3, 1) == 3
assert min_days([1, 10, 3, 10, 2], 3, 2) == -1
assert min_days([7, 7, 7, 7, 12, 7, 7], 2, 3) == 12
print(min_days([1, 10, 3, 10, 2], 3, 1))

### 3.3.5 [1283. Find the Smallest Divisor Given a Threshold](https://leetcode.com/problems/find-the-smallest-divisor-given-a-threshold/) — ★ · Medium

**Problem.** Smallest divisor `d` such that `sum(ceil(x / d)) ≤ threshold`.  
**Signal.** Same shape as Koko: monotone cost in `d`.  
**Idea.** Binary search `d` in `[1, max(nums)]`.  
**Cost.** O(n log max)

In [ ]:
def smallest_divisor(nums: list[int], threshold: int) -> int:
    """Smallest divisor keeping the rounded-up sum within threshold."""
    lo, hi = 1, max(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if sum((x + mid - 1) // mid for x in nums) <= threshold:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert smallest_divisor([1, 2, 5, 9], 6) == 5
assert smallest_divisor([44, 22, 33, 11, 1], 5) == 44
print(smallest_divisor([1, 2, 5, 9], 6))

### 3.3.6 [378. Kth Smallest Element in a Sorted Matrix](https://leetcode.com/problems/kth-smallest-element-in-a-sorted-matrix/) — ★ · Medium

**Problem.** k-th smallest value in an `n × n` matrix whose rows and columns are sorted.  
**Signal.** Binary search on the *value*, counting elements ≤ value with a staircase walk.  
**Idea.** For a candidate `v`, count entries ≤ `v` in O(n) starting from the bottom-left corner. The first `v` with count ≥ `k` is the answer (it is always a matrix value).  
**Cost.** O(n log(max − min))

In [ ]:
def kth_smallest_matrix(matrix: list[list[int]], k: int) -> int:
    """k-th smallest entry of a sorted matrix."""
    n = len(matrix)

    def count_le(v: int) -> int:
        r, c, count = n - 1, 0, 0
        while r >= 0 and c < n:
            if matrix[r][c] <= v:
                count += r + 1
                c += 1
            else:
                r -= 1
        return count

    lo, hi = matrix[0][0], matrix[-1][-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_le(mid) >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo


assert kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8) == 13
assert kth_smallest_matrix([[-5]], 1) == -5
print(kth_smallest_matrix([[1, 5, 9], [10, 11, 13], [12, 13, 15]], 8))

### 3.3.7 [4. Median of Two Sorted Arrays](https://leetcode.com/problems/median-of-two-sorted-arrays/) — ★★ · Hard

**Problem.** Median of two sorted arrays in O(log(min(m, n))).  
**Signal.** Binary search a *partition* of the shorter array.  
**Idea.** Choose `i` elements from `a` and `j = half − i` from `b` for the left half. The partition is right when `a[i−1] ≤ b[j]` and `b[j−1] ≤ a[i]`. If `a[i−1] > b[j]`, take fewer from `a`; otherwise more. Pad with ±∞ at the edges. The median comes from the max of the left side and min of the right side.  
**Cost.** O(log min(m, n))

In [ ]:
def find_median_sorted_arrays(a: list[int], b: list[int]) -> float:
    """Median of the union of two sorted arrays."""
    if len(a) > len(b):
        a, b = b, a
    m, n = len(a), len(b)
    half = (m + n + 1) // 2
    lo, hi = 0, m
    inf = float("inf")
    while True:
        i = (lo + hi) // 2
        j = half - i
        a_left = a[i - 1] if i > 0 else -inf
        a_right = a[i] if i < m else inf
        b_left = b[j - 1] if j > 0 else -inf
        b_right = b[j] if j < n else inf
        if a_left > b_right:
            hi = i - 1
        elif b_left > a_right:
            lo = i + 1
        else:
            left_max = max(a_left, b_left)
            if (m + n) % 2:
                return float(left_max)
            return (left_max + min(a_right, b_right)) / 2


assert find_median_sorted_arrays([1, 3], [2]) == 2.0
assert find_median_sorted_arrays([1, 2], [3, 4]) == 2.5
assert find_median_sorted_arrays([], [1]) == 1.0
print(find_median_sorted_arrays([1, 2], [3, 4]))

### 3.3.8 [981. Time Based Key-Value Store](https://leetcode.com/problems/time-based-key-value-store/) — ★★ · Medium

**Problem.** `set(key, value, t)`; `get(key, t)` returns the value with the largest timestamp ≤ `t`.  
**Signal.** Timestamps arrive increasing → per-key sorted list + upper-bound search.  
**Idea.** Store `key → ([times], [values])`. `get` finds `bisect_right(times, t) − 1`; if it is −1, return `""`.  
**Cost.** O(1) set · O(log n) get

In [ ]:
class TimeMap:
    """Versioned key-value store."""

    def __init__(self) -> None:
        self.times: dict[str, list[int]] = defaultdict(list)
        self.values: dict[str, list[str]] = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.times[key].append(timestamp)
        self.values[key].append(value)

    def get(self, key: str, timestamp: int) -> str:
        i = bisect_right(self.times[key], timestamp) - 1
        return self.values[key][i] if i >= 0 else ""


tm = TimeMap()
tm.set("foo", "bar", 1)
assert tm.get("foo", 1) == "bar" and tm.get("foo", 3) == "bar"
tm.set("foo", "bar2", 4)
assert tm.get("foo", 4) == "bar2" and tm.get("foo", 5) == "bar2"
assert tm.get("foo", 0) == ""
print(tm.get("foo", 3))

---
# 4) Linked List

Three tools solve nearly every linked-list question:

| Tool | Use it for |
|---|---|
| **Dummy head** `dummy = ListNode(0, head)` | any operation that might change or delete the head |
| **Slow/fast pointers** | middle node, cycle detection, k-th from the end |
| **In-place reversal** `prev, cur = None, head` | reverse all / part / k-groups, palindrome check, reorder |

Draw three nodes on paper and update pointers in this order: save `next`, rewire, advance.
The helper cell below converts between Python lists and linked lists for the tests.

In [ ]:
class ListNode:
    """Singly linked list node."""

    def __init__(self, val: int = 0, next: "ListNode | None" = None) -> None:
        self.val = val
        self.next = next


def build(values: list[int]) -> ListNode | None:
    """Linked list from a Python list."""
    dummy = tail = ListNode()
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: ListNode | None) -> list[int]:
    """Python list from a linked list."""
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out


print(to_list(build([1, 2, 3])))

## 4.1 Reversal and merging

### 4.1.1 [206. Reverse Linked List](https://leetcode.com/problems/reverse-linked-list/) — ★★★ · Easy

**Problem.** Reverse a singly linked list.  
**Signal.** The fundamental pointer rewiring.  
**Idea.** Walk the list with `prev` and `cur`; point `cur.next` back to `prev`. Save `cur.next` first so you can continue.  
**Cost.** O(n) time · O(1) space

In [ ]:
def reverse_list(head: ListNode | None) -> ListNode | None:
    """Head of the reversed list."""
    prev = None
    cur = head
    while cur:
        nxt = cur.next
        cur.next = prev
        prev, cur = cur, nxt
    return prev


assert to_list(reverse_list(build([1, 2, 3, 4, 5]))) == [5, 4, 3, 2, 1]
assert to_list(reverse_list(build([]))) == []
print(to_list(reverse_list(build([1, 2]))))

### 4.1.2 [21. Merge Two Sorted Lists](https://leetcode.com/problems/merge-two-sorted-lists/) — ★★★ · Easy

**Problem.** Merge two sorted lists into one sorted list.  
**Signal.** Merge step of merge sort + dummy head.  
**Idea.** Attach the smaller head to `tail` and advance that list; at the end attach whatever remains.  
**Cost.** O(m + n) time · O(1) space

In [ ]:
def merge_two_lists(a: ListNode | None, b: ListNode | None) -> ListNode | None:
    """Merged sorted list."""
    dummy = tail = ListNode()
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next


assert to_list(merge_two_lists(build([1, 2, 4]), build([1, 3, 4]))) == [
    1,
    1,
    2,
    3,
    4,
    4,
]
assert to_list(merge_two_lists(build([]), build([0]))) == [0]
print(to_list(merge_two_lists(build([1, 2, 4]), build([1, 3, 4]))))

### 4.1.3 [23. Merge k Sorted Lists](https://leetcode.com/problems/merge-k-sorted-lists/) — ★★★ · Hard

**Problem.** Merge `k` sorted linked lists.  
**Signal.** Repeatedly need the smallest of k heads → min-heap.  
**Idea.** Push each list's head as `(val, index, node)` — the index breaks ties because nodes are not comparable. Pop the smallest, append it, push its successor.  
**Cost.** O(N log k) for N total nodes · O(k) space

In [ ]:
def merge_k_lists(lists: list[ListNode | None]) -> ListNode | None:
    """Merged sorted list of k sorted lists."""
    heap = [(node.val, i, node) for i, node in enumerate(lists) if node]
    heapq.heapify(heap)
    dummy = tail = ListNode()
    while heap:
        _, i, node = heapq.heappop(heap)
        tail.next = tail = node
        if node.next:
            heapq.heappush(heap, (node.next.val, i, node.next))
    return dummy.next


lists = [build([1, 4, 5]), build([1, 3, 4]), build([2, 6])]
assert to_list(merge_k_lists(lists)) == [1, 1, 2, 3, 4, 4, 5, 6]
assert to_list(merge_k_lists([])) == []
print("ok")

### 4.1.4 [92. Reverse Linked List II](https://leetcode.com/problems/reverse-linked-list-ii/) — ★★ · Medium

**Problem.** Reverse the nodes from position `left` to `right` (1-indexed) in one pass.  
**Signal.** Partial reversal → dummy head + "move next node to the front of the sublist".  
**Idea.** Walk `pre` to the node before `left`. Then `right − left` times, take the node after `cur` and insert it right after `pre`. The sublist reverses in place.  
**Cost.** O(n) time · O(1) space

In [ ]:
def reverse_between(head: ListNode | None, left: int, right: int) -> ListNode | None:
    """Reverse positions left..right."""
    dummy = ListNode(0, head)
    pre = dummy
    for _ in range(left - 1):
        pre = pre.next
    cur = pre.next
    for _ in range(right - left):
        moved = cur.next
        cur.next = moved.next
        moved.next = pre.next
        pre.next = moved
    return dummy.next


assert to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)) == [1, 4, 3, 2, 5]
assert to_list(reverse_between(build([5]), 1, 1)) == [5]
print(to_list(reverse_between(build([1, 2, 3, 4, 5]), 2, 4)))

### 4.1.5 [25. Reverse Nodes in k-Group](https://leetcode.com/problems/reverse-nodes-in-k-group/) — ★★ · Hard

**Problem.** Reverse every consecutive group of `k` nodes; a final short group stays as is.  
**Signal.** Repeated partial reversal with a "group previous" pointer.  
**Idea.** From `group_prev`, check that `k` nodes exist. Reverse those `k` nodes, then reconnect: `group_prev.next` becomes the new group head, and the old group head (now the tail) becomes the next `group_prev`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def reverse_k_group(head: ListNode | None, k: int) -> ListNode | None:
    """Reverse nodes in groups of k."""
    dummy = ListNode(0, head)
    group_prev = dummy
    while True:
        kth = group_prev
        for _ in range(k):
            kth = kth.next
            if not kth:
                return dummy.next
        group_next = kth.next
        prev, cur = group_next, group_prev.next
        while cur is not group_next:
            nxt = cur.next
            cur.next = prev
            prev, cur = cur, nxt
        old_head = group_prev.next
        group_prev.next = kth
        group_prev = old_head


assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 2)) == [2, 1, 4, 3, 5]
assert to_list(reverse_k_group(build([1, 2, 3, 4, 5]), 3)) == [3, 2, 1, 4, 5]
print(to_list(reverse_k_group(build([1, 2, 3, 4, 5, 6]), 3)))

### 4.1.6 [24. Swap Nodes in Pairs](https://leetcode.com/problems/swap-nodes-in-pairs/) — ★★ · Medium

**Problem.** Swap every two adjacent nodes (rewire nodes, not values).  
**Signal.** Reverse k-Group with `k = 2`, small enough to write directly.  
**Idea.** With `prev` before the pair `a, b`: `prev → b → a → rest`, then `prev = a`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def swap_pairs(head: ListNode | None) -> ListNode | None:
    """Swap adjacent pairs of nodes."""
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next and prev.next.next:
        a, b = prev.next, prev.next.next
        prev.next, a.next, b.next = b, b.next, a
        prev = a
    return dummy.next


assert to_list(swap_pairs(build([1, 2, 3, 4]))) == [2, 1, 4, 3]
assert to_list(swap_pairs(build([1, 2, 3]))) == [2, 1, 3]
print(to_list(swap_pairs(build([1, 2, 3, 4]))))

### 4.1.7 [2. Add Two Numbers](https://leetcode.com/problems/add-two-numbers/) — ★★ · Medium

**Problem.** Digits stored in reverse order; return the sum as a list.  
**Signal.** Grade-school addition from the least significant digit, carry propagates.  
**Idea.** Walk both lists together, `total = a + b + carry`, emit `total % 10`, carry `total // 10`. Continue while either list or the carry remains.  
**Cost.** O(max(m, n)) time

In [ ]:
def add_two_numbers(a: ListNode | None, b: ListNode | None) -> ListNode | None:
    """Sum of two reversed-digit numbers."""
    dummy = tail = ListNode()
    carry = 0
    while a or b or carry:
        total = carry + (a.val if a else 0) + (b.val if b else 0)
        carry, digit = divmod(total, 10)
        tail.next = tail = ListNode(digit)
        a = a.next if a else None
        b = b.next if b else None
    return dummy.next


assert to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))) == [7, 0, 8]
assert to_list(add_two_numbers(build([9] * 7), build([9] * 4))) == [
    8,
    9,
    9,
    9,
    0,
    0,
    0,
    1,
]
print(to_list(add_two_numbers(build([2, 4, 3]), build([5, 6, 4]))))

### 4.1.8 [445. Add Two Numbers II](https://leetcode.com/problems/add-two-numbers-ii/) — ★ · Medium

**Problem.** Same as Add Two Numbers, but digits are stored most-significant first.  
**Signal.** Need digits from the end → stacks (or reverse both lists).  
**Idea.** Push both lists' digits onto stacks, pop and add with carry, and prepend each new digit node to the result.  
**Cost.** O(m + n) time · O(m + n) space

In [ ]:
def add_two_numbers_ii(a: ListNode | None, b: ListNode | None) -> ListNode | None:
    """Sum of two forward-digit numbers."""
    s1, s2 = to_list(a), to_list(b)
    head, carry = None, 0
    while s1 or s2 or carry:
        total = carry + (s1.pop() if s1 else 0) + (s2.pop() if s2 else 0)
        carry, digit = divmod(total, 10)
        head = ListNode(digit, head)
    return head


assert to_list(add_two_numbers_ii(build([7, 2, 4, 3]), build([5, 6, 4]))) == [
    7,
    8,
    0,
    7,
]
assert to_list(add_two_numbers_ii(build([0]), build([0]))) == [0]
print(to_list(add_two_numbers_ii(build([7, 2, 4, 3]), build([5, 6, 4]))))

### 4.1.9 [148. Sort List](https://leetcode.com/problems/sort-list/) — ★★ · Medium

**Problem.** Sort a linked list in O(n log n).  
**Signal.** Linked lists favour merge sort (no random access needed).  
**Idea.** Split at the middle with slow/fast pointers, sort each half recursively, merge with Merge Two Sorted Lists. (A bottom-up version achieves O(1) extra space.)  
**Cost.** O(n log n) time · O(log n) stack

In [ ]:
def sort_list(head: ListNode | None) -> ListNode | None:
    """Merge-sorted list."""
    if not head or not head.next:
        return head
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    right = slow.next
    slow.next = None
    return merge_two_lists(sort_list(head), sort_list(right))


assert to_list(sort_list(build([4, 2, 1, 3]))) == [1, 2, 3, 4]
assert to_list(sort_list(build([-1, 5, 3, 4, 0]))) == [-1, 0, 3, 4, 5]
print(to_list(sort_list(build([4, 2, 1, 3]))))

## 4.2 Slow and fast pointers

### 4.2.1 [876. Middle of the Linked List](https://leetcode.com/problems/middle-of-the-linked-list/) — ★★ · Easy

**Problem.** Return the middle node (second middle if even length).  
**Signal.** Fast moves two steps per slow step → slow ends in the middle.  
**Idea.** When `fast` reaches the end, `slow` has covered half the distance.  
**Cost.** O(n) time · O(1) space

In [ ]:
def middle_node(head: ListNode | None) -> ListNode | None:
    """Middle node (second of two middles)."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    return slow


assert to_list(middle_node(build([1, 2, 3, 4, 5]))) == [3, 4, 5]
assert to_list(middle_node(build([1, 2, 3, 4, 5, 6]))) == [4, 5, 6]
print(middle_node(build([1, 2, 3])).val)

### 4.2.2 [141. Linked List Cycle](https://leetcode.com/problems/linked-list-cycle/) — ★★★ · Easy

**Problem.** Does the list contain a cycle? O(1) memory.  
**Signal.** Floyd's tortoise and hare.  
**Idea.** If there is a cycle, the fast pointer gains one node per step on the slow pointer inside the loop and must land on it. Without a cycle, fast reaches `None`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def has_cycle(head: ListNode | None) -> bool:
    """True if the list loops."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False


def build_cycle(values: list[int], pos: int) -> ListNode | None:
    """List whose tail links back to index pos (-1 for none)."""
    head = build(values)
    if pos >= 0:
        nodes = []
        node = head
        while node:
            nodes.append(node)
            node = node.next
        nodes[-1].next = nodes[pos]
    return head


assert has_cycle(build_cycle([3, 2, 0, -4], 1))
assert not has_cycle(build_cycle([1], -1))
print(has_cycle(build_cycle([1, 2], 0)))

### 4.2.3 [142. Linked List Cycle II](https://leetcode.com/problems/linked-list-cycle-ii/) — ★ · Medium

**Problem.** Return the node where the cycle begins, or `None`.  
**Signal.** Floyd phase 2: after meeting, a pointer from the head and one from the meeting point meet at the cycle start.  
**Idea.** If the tail before the cycle has length `a` and they meet `b` into the cycle of length `c`, then `2(a + b) = a + b + kc`, so `a = kc − b`: walking `a` steps from the meeting point lands exactly on the cycle start.  
**Cost.** O(n) time · O(1) space

In [ ]:
def detect_cycle(head: ListNode | None) -> ListNode | None:
    """First node of the cycle, if any."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            finder = head
            while finder is not slow:
                finder, slow = finder.next, slow.next
            return finder
    return None


assert detect_cycle(build_cycle([3, 2, 0, -4], 1)).val == 2
assert detect_cycle(build_cycle([1], -1)) is None
print(detect_cycle(build_cycle([1, 2], 0)).val)

### 4.2.4 [287. Find the Duplicate Number](https://leetcode.com/problems/find-the-duplicate-number/) — ★★ · Medium

**Problem.** `n + 1` integers in `1..n`, one value repeated; find it with O(1) space, no modification.  
**Signal.** Treat `i → nums[i]` as a linked list; the duplicate is the cycle entrance.  
**Idea.** Two indices point to the same value, so the functional graph has a cycle whose entry is the duplicate. Run Floyd's algorithm on `x = nums[x]` exactly as in Cycle II.  
**Cost.** O(n) time · O(1) space

In [ ]:
def find_duplicate(nums: list[int]) -> int:
    """The repeated value."""
    slow = fast = 0
    while True:
        slow, fast = nums[slow], nums[nums[fast]]
        if slow == fast:
            break
    finder = 0
    while finder != slow:
        finder, slow = nums[finder], nums[slow]
    return finder


assert find_duplicate([1, 3, 4, 2, 2]) == 2
assert find_duplicate([3, 1, 3, 4, 2]) == 3
assert find_duplicate([3, 3, 3, 3, 3]) == 3
print(find_duplicate([1, 3, 4, 2, 2]))

### 4.2.5 [19. Remove Nth Node From End of List](https://leetcode.com/problems/remove-nth-node-from-end-of-list/) — ★★★ · Medium

**Problem.** Remove the n-th node from the end in one pass.  
**Signal.** Gap of n between two pointers.  
**Idea.** Move `fast` n steps ahead of `slow` (starting at a dummy). Advance both until `fast` is the last node; `slow` is then just before the node to remove.  
**Cost.** O(L) time · O(1) space

In [ ]:
def remove_nth_from_end(head: ListNode | None, n: int) -> ListNode | None:
    """List without its n-th node from the end."""
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):
        fast = fast.next
    while fast.next:
        slow, fast = slow.next, fast.next
    slow.next = slow.next.next
    return dummy.next


assert to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)) == [1, 2, 3, 5]
assert to_list(remove_nth_from_end(build([1]), 1)) == []
assert to_list(remove_nth_from_end(build([1, 2]), 2)) == [2]
print(to_list(remove_nth_from_end(build([1, 2, 3, 4, 5]), 2)))

### 4.2.6 [234. Palindrome Linked List](https://leetcode.com/problems/palindrome-linked-list/) — ★★ · Easy

**Problem.** Is the list a palindrome? O(1) space.  
**Signal.** Middle + reverse second half + compare.  
**Idea.** Find the middle with slow/fast, reverse the second half, walk both halves comparing values. (Restore the list afterwards if the caller cares.)  
**Cost.** O(n) time · O(1) space

In [ ]:
def is_palindrome_list(head: ListNode | None) -> bool:
    """True if values read the same both ways."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    left, right = head, reverse_list(slow)
    while right:
        if left.val != right.val:
            return False
        left, right = left.next, right.next
    return True


assert is_palindrome_list(build([1, 2, 2, 1]))
assert not is_palindrome_list(build([1, 2]))
assert is_palindrome_list(build([1, 2, 3, 2, 1]))
print(is_palindrome_list(build([1, 2, 2, 1])))

### 4.2.7 [143. Reorder List](https://leetcode.com/problems/reorder-list/) — ★★★ · Medium

**Problem.** Reorder `L0 → L1 → … → Ln` into `L0 → Ln → L1 → Ln−1 → …` in place.  
**Signal.** Middle + reverse second half + interleave merge.  
**Idea.** Split at the middle, reverse the second half, then alternately take one node from each half.  
**Cost.** O(n) time · O(1) space

In [ ]:
def reorder_list(head: ListNode | None) -> None:
    """Interleave front and reversed back halves in place."""
    slow, fast = head, head.next
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    second = reverse_list(slow.next)
    slow.next = None
    first = head
    while second:
        f_next, s_next = first.next, second.next
        first.next, second.next = second, f_next
        first, second = f_next, s_next


h = build([1, 2, 3, 4, 5])
reorder_list(h)
assert to_list(h) == [1, 5, 2, 4, 3]
h = build([1, 2, 3, 4])
reorder_list(h)
assert to_list(h) == [1, 4, 2, 3]
print(to_list(h))

### 4.2.8 [160. Intersection of Two Linked Lists](https://leetcode.com/problems/intersection-of-two-linked-lists/) — ★★ · Easy

**Problem.** Node where two lists merge, or `None`; O(1) space.  
**Signal.** Equalise path lengths by switching heads.  
**Idea.** Walk pointer `a` over list A then list B, and `b` over B then A. Both travel `len(A) + len(B)` steps and meet at the intersection (or both reach `None` together).  
**Cost.** O(m + n) time · O(1) space

In [ ]:
def get_intersection_node(head_a: ListNode | None, head_b: ListNode | None):
    """Shared node of two lists, or None."""
    a, b = head_a, head_b
    while a is not b:
        a = a.next if a else head_b
        b = b.next if b else head_a
    return a


shared = build([8, 4, 5])
la = ListNode(4, ListNode(1, shared))
lb = ListNode(5, ListNode(6, ListNode(1, shared)))
assert get_intersection_node(la, lb) is shared
assert get_intersection_node(build([1]), build([2])) is None
print(get_intersection_node(la, lb).val)

### 4.2.9 [2130. Maximum Twin Sum of a Linked List](https://leetcode.com/problems/maximum-twin-sum-of-a-linked-list/) — ★ · Medium

**Problem.** Even length; twin of node `i` is node `n−1−i`. Max twin sum?  
**Signal.** Middle + reverse second half + walk both halves.  
**Idea.** Reverse the second half; pair the i-th node of each half and take the max sum.  
**Cost.** O(n) time · O(1) space

In [ ]:
def pair_sum(head: ListNode | None) -> int:
    """Largest sum of twin nodes."""
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
    left, right, best = head, reverse_list(slow), 0
    while right:
        best = max(best, left.val + right.val)
        left, right = left.next, right.next
    return best


assert pair_sum(build([5, 4, 2, 1])) == 6
assert pair_sum(build([4, 2, 2, 3])) == 7
print(pair_sum(build([1, 100000])))

## 4.3 Rewiring and deletion

### 4.3.1 [83. Remove Duplicates from Sorted List](https://leetcode.com/problems/remove-duplicates-from-sorted-list/) — ★ · Easy

**Problem.** Delete duplicates so each value appears once.  
**Signal.** Sorted → duplicates are adjacent.  
**Idea.** If `cur.next` has the same value, skip it; otherwise advance.  
**Cost.** O(n) time · O(1) space

In [ ]:
def delete_duplicates(head: ListNode | None) -> ListNode | None:
    """Sorted list with each value once."""
    cur = head
    while cur and cur.next:
        if cur.next.val == cur.val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return head


assert to_list(delete_duplicates(build([1, 1, 2, 3, 3]))) == [1, 2, 3]
print(to_list(delete_duplicates(build([1, 1, 2]))))

### 4.3.2 [82. Remove Duplicates from Sorted List II](https://leetcode.com/problems/remove-duplicates-from-sorted-list-ii/) — ★★ · Medium

**Problem.** Delete *all* nodes that have duplicate values, keeping only distinct values.  
**Signal.** Head may be deleted → dummy; skip whole runs.  
**Idea.** `prev` is the last kept node. If `prev.next` starts a run of equal values, skip the whole run; otherwise advance `prev`.  
**Cost.** O(n) time · O(1) space

In [ ]:
def delete_duplicates_ii(head: ListNode | None) -> ListNode | None:
    """Sorted list keeping only values that appear once."""
    dummy = ListNode(0, head)
    prev = dummy
    while prev.next:
        cur = prev.next
        if cur.next and cur.next.val == cur.val:
            while cur.next and cur.next.val == cur.val:
                cur = cur.next
            prev.next = cur.next
        else:
            prev = cur
    return dummy.next


assert to_list(delete_duplicates_ii(build([1, 2, 3, 3, 4, 4, 5]))) == [1, 2, 5]
assert to_list(delete_duplicates_ii(build([1, 1, 1, 2, 3]))) == [2, 3]
print(to_list(delete_duplicates_ii(build([1, 2, 3, 3, 4, 4, 5]))))

### 4.3.3 [203. Remove Linked List Elements](https://leetcode.com/problems/remove-linked-list-elements/) — ★ · Easy

**Problem.** Remove all nodes with value `val`.  
**Signal.** Head may be removed → dummy head.  
**Idea.** From the dummy, skip `cur.next` while it holds `val`; otherwise advance.  
**Cost.** O(n) time · O(1) space

In [ ]:
def remove_elements(head: ListNode | None, val: int) -> ListNode | None:
    """List without nodes equal to val."""
    dummy = cur = ListNode(0, head)
    while cur.next:
        if cur.next.val == val:
            cur.next = cur.next.next
        else:
            cur = cur.next
    return dummy.next


assert to_list(remove_elements(build([1, 2, 6, 3, 4, 5, 6]), 6)) == [1, 2, 3, 4, 5]
assert to_list(remove_elements(build([7, 7, 7]), 7)) == []
print(to_list(remove_elements(build([1, 2, 6, 3]), 6)))

### 4.3.4 [61. Rotate List](https://leetcode.com/problems/rotate-list/) — ★★ · Medium

**Problem.** Rotate the list to the right by `k` places.  
**Signal.** Rotation = close into a ring, then cut at the right place.  
**Idea.** Find the length `n` and tail; link tail to head. The new tail is `n − k % n − 1` steps from the head; cut after it.  
**Cost.** O(n) time · O(1) space

In [ ]:
def rotate_right(head: ListNode | None, k: int) -> ListNode | None:
    """List rotated right by k."""
    if not head:
        return head
    n, tail = 1, head
    while tail.next:
        tail, n = tail.next, n + 1
    tail.next = head
    new_tail = head
    for _ in range(n - k % n - 1):
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None
    return new_head


assert to_list(rotate_right(build([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]
assert to_list(rotate_right(build([0, 1, 2]), 4)) == [2, 0, 1]
print(to_list(rotate_right(build([1, 2, 3, 4, 5]), 2)))

### 4.3.5 [86. Partition List](https://leetcode.com/problems/partition-list/) — ★★ · Medium

**Problem.** Move nodes `< x` before nodes `≥ x`, preserving relative order.  
**Signal.** Stable partition → build two lists with two dummies, then join.  
**Idea.** Append each node to `before` or `after`; terminate `after` with `None` (important, or a cycle may remain) and link `before`'s tail to `after`'s head.  
**Cost.** O(n) time · O(1) space

In [ ]:
def partition(head: ListNode | None, x: int) -> ListNode | None:
    """Stable partition around x."""
    before = b_tail = ListNode()
    after = a_tail = ListNode()
    while head:
        if head.val < x:
            b_tail.next = b_tail = head
        else:
            a_tail.next = a_tail = head
        head = head.next
    a_tail.next = None
    b_tail.next = after.next
    return before.next


assert to_list(partition(build([1, 4, 3, 2, 5, 2]), 3)) == [1, 2, 2, 4, 3, 5]
assert to_list(partition(build([2, 1]), 2)) == [1, 2]
print(to_list(partition(build([1, 4, 3, 2, 5, 2]), 3)))

### 4.3.6 [328. Odd Even Linked List](https://leetcode.com/problems/odd-even-linked-list/) — ★★ · Medium

**Problem.** Group all odd-positioned nodes, then even-positioned nodes, in place.  
**Signal.** Two interleaved chains, joined at the end.  
**Idea.** Keep `odd` and `even` tails; each step, `odd.next = even.next` and `even.next = odd.next`. Finally attach the even chain's head after the odd tail.  
**Cost.** O(n) time · O(1) space

In [ ]:
def odd_even_list(head: ListNode | None) -> ListNode | None:
    """Odd positions followed by even positions."""
    if not head:
        return head
    odd, even = head, head.next
    even_head = even
    while even and even.next:
        odd.next = even.next
        odd = odd.next
        even.next = odd.next
        even = even.next
    odd.next = even_head
    return head


assert to_list(odd_even_list(build([1, 2, 3, 4, 5]))) == [1, 3, 5, 2, 4]
assert to_list(odd_even_list(build([2, 1, 3, 5, 6, 4, 7]))) == [2, 3, 6, 7, 1, 5, 4]
print(to_list(odd_even_list(build([1, 2, 3, 4, 5]))))

### 4.3.7 [138. Copy List with Random Pointer](https://leetcode.com/problems/copy-list-with-random-pointer/) — ★★ · Medium

**Problem.** Deep-copy a list where each node also has a `random` pointer.  
**Signal.** Map old node → new node, then wire pointers.  
**Idea.** First pass creates a copy of every node in a dict (with `None → None`). Second pass sets `copy.next = map[old.next]` and `copy.random = map[old.random]`. (O(1) space variant: interleave copies into the original list.)  
**Cost.** O(n) time · O(n) space

In [ ]:
class RandomNode:
    """Node with an extra random pointer."""

    def __init__(self, val: int) -> None:
        self.val = val
        self.next: "RandomNode | None" = None
        self.random: "RandomNode | None" = None


def copy_random_list(head: RandomNode | None) -> RandomNode | None:
    """Deep copy preserving random links."""
    copies: dict = {None: None}
    node = head
    while node:
        copies[node] = RandomNode(node.val)
        node = node.next
    node = head
    while node:
        copies[node].next = copies[node.next]
        copies[node].random = copies[node.random]
        node = node.next
    return copies[head]


nodes = [RandomNode(v) for v in [7, 13, 11, 10, 1]]
for a, b in zip(nodes, nodes[1:]):
    a.next = b
for node, r in zip(nodes, [None, 0, 4, 2, 0]):
    node.random = nodes[r] if r is not None else None
copy = copy_random_list(nodes[0])
out = []
while copy:
    out.append((copy.val, copy.random.val if copy.random else None))
    assert copy not in nodes
    copy = copy.next
assert out == [(7, None), (13, 7), (11, 1), (10, 11), (1, 7)]
print(out)

## 4.4 Linked-list design

### 4.4.1 [146. LRU Cache](https://leetcode.com/problems/lru-cache/) — ★★ · Medium

**Problem.** Fixed-capacity cache with O(1) `get` and `put`, evicting the least recently used key.  
**Signal.** O(1) lookup + O(1) recency order → hash map + doubly linked list.  
**Idea.** The map gives key → node. The list keeps nodes from least to most recent between two sentinels. `get` moves the node to the end; `put` inserts/updates at the end and, if over capacity, removes the node after the head sentinel. (In Python, `OrderedDict` with `move_to_end` does the same, but interviewers usually want the list.)  
**Cost.** O(1) per operation · O(capacity) space

In [ ]:
class DNode:
    """Doubly linked node for the cache."""

    def __init__(self, key: int = 0, val: int = 0) -> None:
        self.key, self.val = key, val
        self.prev: "DNode | None" = None
        self.next: "DNode | None" = None


class LRUCache:
    """Least-recently-used cache."""

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.map: dict[int, DNode] = {}
        self.head, self.tail = DNode(), DNode()
        self.head.next, self.tail.prev = self.tail, self.head

    def _remove(self, node: DNode) -> None:
        node.prev.next, node.next.prev = node.next, node.prev

    def _append(self, node: DNode) -> None:
        node.prev, node.next = self.tail.prev, self.tail
        self.tail.prev.next = node
        self.tail.prev = node

    def get(self, key: int) -> int:
        if key not in self.map:
            return -1
        node = self.map[key]
        self._remove(node)
        self._append(node)
        return node.val

    def put(self, key: int, value: int) -> None:
        if key in self.map:
            self._remove(self.map[key])
        node = DNode(key, value)
        self.map[key] = node
        self._append(node)
        if len(self.map) > self.capacity:
            lru = self.head.next
            self._remove(lru)
            del self.map[lru.key]


cache = LRUCache(2)
cache.put(1, 1)
cache.put(2, 2)
assert cache.get(1) == 1
cache.put(3, 3)
assert cache.get(2) == -1
cache.put(4, 4)
assert [cache.get(1), cache.get(3), cache.get(4)] == [-1, 3, 4]
print("ok")

### 4.4.2 [460. LFU Cache](https://leetcode.com/problems/lfu-cache/) — ★ · Hard

**Problem.** Cache evicting the least *frequently* used key (ties → least recently used), O(1) ops.  
**Signal.** Frequency buckets, each an ordered set of keys, plus a `min_freq` pointer.  
**Idea.** `key → (value, freq)` and `freq → OrderedDict of keys` (insertion order = recency). Touching a key moves it from bucket `f` to `f + 1`; if bucket `min_freq` empties, bump `min_freq`. A new key enters bucket 1 and resets `min_freq = 1`. Evict the first key of bucket `min_freq`.  
**Cost.** O(1) per operation

In [ ]:
class LFUCache:
    """Least-frequently-used cache with LRU tie-break."""

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.entries: dict[int, list[int]] = {}
        self.buckets: dict[int, OrderedDict] = defaultdict(OrderedDict)
        self.min_freq = 0

    def _touch(self, key: int) -> None:
        value, freq = self.entries[key]
        del self.buckets[freq][key]
        if not self.buckets[freq] and self.min_freq == freq:
            self.min_freq += 1
        self.entries[key] = [value, freq + 1]
        self.buckets[freq + 1][key] = None

    def get(self, key: int) -> int:
        if key not in self.entries:
            return -1
        self._touch(key)
        return self.entries[key][0]

    def put(self, key: int, value: int) -> None:
        if self.capacity == 0:
            return
        if key in self.entries:
            self.entries[key][0] = value
            self._touch(key)
            return
        if len(self.entries) == self.capacity:
            old, _ = self.buckets[self.min_freq].popitem(last=False)
            del self.entries[old]
        self.entries[key] = [value, 1]
        self.buckets[1][key] = None
        self.min_freq = 1


lfu = LFUCache(2)
lfu.put(1, 1)
lfu.put(2, 2)
assert lfu.get(1) == 1
lfu.put(3, 3)
assert lfu.get(2) == -1 and lfu.get(3) == 3
lfu.put(4, 4)
assert [lfu.get(1), lfu.get(3), lfu.get(4)] == [-1, 3, 4]
print("ok")

### 4.4.3 [1472. Design Browser History](https://leetcode.com/problems/design-browser-history/) — ★ · Medium

**Problem.** `visit(url)` clears forward history; `back(steps)` and `forward(steps)` move and return the current URL.  
**Signal.** A list plus a cursor and a "last valid" index (a doubly linked list also works).  
**Idea.** Keep `history`, `cur` and `last`. Visiting writes at `cur + 1` and sets `last = cur`, discarding forward pages in O(1). Back/forward clamp the cursor to `[0, last]`.  
**Cost.** O(1) per operation

In [ ]:
class BrowserHistory:
    """Back/forward browser history."""

    def __init__(self, homepage: str) -> None:
        self.history = [homepage]
        self.cur = self.last = 0

    def visit(self, url: str) -> None:
        self.cur += 1
        if self.cur == len(self.history):
            self.history.append(url)
        else:
            self.history[self.cur] = url
        self.last = self.cur

    def back(self, steps: int) -> str:
        self.cur = max(0, self.cur - steps)
        return self.history[self.cur]

    def forward(self, steps: int) -> str:
        self.cur = min(self.last, self.cur + steps)
        return self.history[self.cur]


b = BrowserHistory("leetcode.com")
for url in ["google.com", "facebook.com", "youtube.com"]:
    b.visit(url)
assert b.back(1) == "facebook.com" and b.back(1) == "google.com"
assert b.forward(1) == "facebook.com"
b.visit("linkedin.com")
assert b.forward(2) == "linkedin.com"
assert b.back(2) == "google.com" and b.back(7) == "leetcode.com"
print("ok")

---
# 5) Conclusion

| Pattern | Recognise it by | Core move | Cost |
|---|---|---|---|
| Matching stack | brackets, paths, nested decode, expression | push opens/context, pop on close | O(n) |
| Monotonic stack | next greater/smaller, span, histogram | pop while new element breaks the order | O(n) |
| Contribution counting | sum over all subarrays of min/max | previous/next smaller distances | O(n) |
| Lower-bound binary search | sorted lookup, insert position, first/last | `while lo < hi`, `hi = mid` on true | O(log n) |
| Rotated search | rotated sorted array | one half is always sorted | O(log n) |
| Search on answer | "minimum capacity / speed / day such that…" | binary search the value, greedy feasibility | O(n log R) |
| Partition search | median of two arrays | binary search the cut of the shorter array | O(log min) |
| Dummy head | head may change | `dummy = ListNode(0, head)` | — |
| Slow/fast | middle, cycle, k-th from end, duplicate number | 2× speed or fixed gap | O(n), O(1) |
| Reversal | reverse all/part/groups, palindrome, reorder | save next → rewire → advance | O(n), O(1) |
| Map + doubly linked list | LRU / LFU caches | O(1) lookup + O(1) reorder | O(1) |